# CT4110 – Topic 9 Assignment
## Variational Quantum Eigensolver (VQE)

This exercise uses **Qiskit** to estimate ground-state energies with the Variational Quantum Eigensolver. It follows the same checkpoint structure as the earlier topic exercises.

- **Task 1:** complete a VQE loop for a supplied two-qubit Hamiltonian using a deliberately weak product-state ansatz.
- **Task 2:** reuse the same Hamiltonian with an improved entangling ansatz and compare both VQE results with exact diagonalisation.
- **Advanced Task 3:** trace the conversion of $H_2$ into a qubit Hamiltonian, then use it in VQE.

Complete every `YOUR CODE HERE` section. Run the notebook from top to bottom before submission. Include the required circuits, numerical output, plots, and short written explanations.

### Documentation used in this exercise

- Qiskit [`QuantumCircuit`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit), [`ParameterVector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.ParameterVector), [`SparsePauliOp`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp), and [`Statevector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector)
- SciPy [`minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html) and the derivative-free [`COBYLA`](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-cobyla.html) method
- Qiskit Nature [installation guide](https://qiskit-community.github.io/qiskit-nature/getting_started.html), [electronic-structure tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/01_electronic_structure.html), [`PySCFDriver`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.drivers.PySCFDriver.html), [`FermionicOp`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.operators.FermionicOp.html), and [`JordanWignerMapper`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.mappers.JordanWignerMapper.html)
- PySCF [installation guide](https://pyscf.org/install.html) and [documentation](https://pyscf.org/user.html)

The setup cell installs the chemistry dependencies only when they are missing. If the setup cell installs or upgrades packages, restart the kernel once and run the notebook again.

In [ ]:
# Course setup – run this cell first
import importlib.util
import subprocess
import sys

required = {
    "qiskit": "qiskit>=2.2",
    "qiskit_nature": "qiskit-nature>=0.7.2",
    "pyscf": "pyscf>=2.6",
    "scipy": "scipy>=1.11",
    "matplotlib": "matplotlib>=3.8",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    # Pass each requirement directly to pip. This avoids shell interpretation
    # of characters such as >= in version constraints.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing, "pylatexenc"
    ])
    print("Installation finished. If an import below fails, restart the kernel once.")

import numpy as np
import matplotlib.pyplot as plt
import qiskit
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector

print("Qiskit version:", qiskit.__version__)
print("Setup complete; core imports succeeded.")


## VQE roadmap

For a parameterised trial state $|\psi(\boldsymbol\theta)\rangle$, VQE minimises

$$E(\boldsymbol\theta)=\langle\psi(\boldsymbol\theta)|H|\psi(\boldsymbol\theta)\rangle.$$

The quantum part prepares the trial state and evaluates its energy. A classical optimiser proposes new parameters. The **variational principle** guarantees

$$\langle\psi|H|\psi\rangle\geq E_0,$$

so lowering the measured energy moves the trial state toward the ground state, provided the ansatz can represent it.

In this notebook we use exact statevectors to focus on the VQE workflow. On hardware, the Pauli expectation values would instead be estimated from measurements. Qiskit writes an $n$-qubit Pauli label as $q_{n-1}\ldots q_0$; for example, `IZ` means $Z_0$.

## Task 1 – Complete VQE with a weak ansatz

The Hamiltonian is supplied below. **Do not redefine it.** It is

$$H=-0.0421I+0.1777Z_0+0.1777Z_1-0.2427Z_0Z_1+0.1220X_0X_1.$$

In this task you will deliberately use a **weak product-state ansatz**: one parameterised $R_y$ rotation on each qubit and no entangling gate. This is enough to practise the VQE loop, but it may not represent an entangled ground state. Task 2 will keep this exact Hamiltonian and improve only the ansatz, giving a fair comparison.

New syntax: [`SparsePauliOp.from_list`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp#from_list) constructs the supplied operator from `(Pauli label, coefficient)` pairs; [`assign_parameters`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit#assign_parameters) binds numerical values to a parameterised circuit; and [`Statevector.expectation_value`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector#expectation_value) evaluates $\langle H\rangle$.

In [ ]:
# Supplied Hamiltonian for Tasks 1 and 2 – run this cell; do not change it
shared_hamiltonian = SparsePauliOp.from_list([
    ("II", -0.0421),
    ("IZ",  0.1777),
    ("ZI",  0.1777),
    ("ZZ", -0.2427),
    ("XX",  0.1220),
])

# Both task names refer to the same operator object.
task1_hamiltonian = shared_hamiltonian
task2_hamiltonian = shared_hamiltonian
task1_initial_parameters = np.array([0.25, -0.40])

print("Shared Hamiltonian H =")
print(shared_hamiltonian)
print("Tasks share the same object:", task1_hamiltonian is task2_hamiltonian)
print("Hermitian:", np.allclose(shared_hamiltonian.to_matrix(),
                                 shared_hamiltonian.to_matrix().conj().T))


In [ ]:
# Task 1 checkpoint 1 – build the weak product-state ansatz
#
# Instructions
# 1. Create ParameterVector("theta", 2).
# 2. Create a two-qubit QuantumCircuit.
# 3. Apply ry(theta[0]) to q0 and ry(theta[1]) to q1.
# 4. Do not add an entangling gate: this limitation is intentional.
# 5. Store the circuit as task1_ansatz.
#
# Documentation:
# https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit#ry

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Draw task1_ansatz using output="mpl".
# - Print task1_ansatz.num_parameters; it must equal 2.
# - Confirm that the circuit contains no two-qubit gates.


In [ ]:
# Task 1 checkpoint 2 – define and test the energy function
#
# Instructions
# 1. Define task1_energy(parameter_values).
# 2. Bind values with task1_ansatz.assign_parameters(parameter_values).
# 3. Create a Statevector from the bound circuit.
# 4. Evaluate task1_hamiltonian and return the real component as a float.
# 5. Evaluate the supplied task1_initial_parameters.
#
# Documentation:
# https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector#from_instruction

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the initial parameter vector and initial energy.
# - Verify that the returned energy is a real Python float.

In [ ]:
# Task 1 checkpoint 3 – optimise and inspect convergence
#
# Instructions
# 1. Create task1_energy_history = [].
# 2. Define a recorded objective that calls task1_energy, appends the energy,
#    and returns it.
# 3. Call scipy.optimize.minimize with the supplied initial parameters,
#    method="COBYLA", and options={"maxiter": 200, "tol": 1e-8}.
# 4. Store the result as task1_result.
# 5. Plot energy against objective-function evaluation.
#
# Documentation:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html
# https://docs.scipy.org/doc/scipy/reference/optimize.minimize-cobyla.html

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the initial energy, final energy, optimised parameters, and message.
# - Print whether final energy < initial energy.
# - Show a labelled convergence plot.
#


## Task 2 – Improve the ansatz for the same Hamiltonian

Continue using the supplied Hamiltonian from Task 1:

$$H=-0.0421I+0.1777Z_0+0.1777Z_1-0.2427Z_0Z_1+0.1220X_0X_1.$$

**Do not construct a new Hamiltonian.** Change only the variational state: replace Task 1's two-parameter product state with a deeper six-parameter ansatz containing two entangling layers. Optimise it, calculate the exact ground-state energy, and compare all three values:

1. Task 1 weak-ansatz energy;
2. Task 2 improved-ansatz energy;
3. exact ground-state energy.

This isolates the effect of ansatz expressibility. New syntax: [`SparsePauliOp.to_matrix`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp#to_matrix) creates the matrix, and NumPy [`eigvalsh`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigvalsh.html) returns the eigenvalues of a Hermitian matrix.

### Required Task 2 circuit

Build the improved ansatz with three rotation layers and two entangling gates. Each rotation uses a different parameter:

```text
q0: ──Ry(φ0)────■────Ry(φ2)────X────Ry(φ4)──
                │              │
q1: ──Ry(φ1)────X────Ry(φ3)────■────Ry(φ5)──
```

The first `CX` uses `q0` as control and `q1` as target. The second reverses the direction: `q1` is the control and `q0` is the target. The finished circuit must contain six independent parameters, two on each rotation layer.

In [ ]:
# Task 2 checkpoint 1 – build an improved ansatz for the shared Hamiltonian
#
# Instructions
# 1. Do not create or modify a Hamiltonian. Use task2_hamiltonian, which is
#    the same object as task1_hamiltonian.
# 2. Define build_task2_ansatz() returning a two-qubit circuit with six
#    distinct ry parameters.
# 3. Use three rotation layers. Place cx(0, 1) after the first layer and
#    cx(1, 0) after the second layer.
# 4. Store the returned circuit as task2_ansatz.
#
# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print whether task1_hamiltonian is task2_hamiltonian; it must be True.
# - Draw task2_ansatz using output="mpl".
# - Print its parameter count; it must equal 6.


In [ ]:
# Task 2 checkpoint 2 – optimise and calculate the exact reference
#
# Instructions
# 1. Define an energy objective for task2_ansatz and task2_hamiltonian.
# 2. Record every objective evaluation in task2_energy_history.
# 3. Use np.random.default_rng(9) to create six reproducible initial values
#    uniformly between -0.25 and 0.25.
# 4. Minimise with COBYLA (maxiter=400, tol=1e-9).
# 5. Convert the Hamiltonian to a matrix and use np.linalg.eigvalsh.
# 6. Calculate abs(Task 2 VQE energy - exact ground energy).
# 7. Compare the Task 1 final energy, Task 2 final energy, and exact energy.
#
# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print Task 1 energy, Task 2 energy, exact energy, absolute error, and
#   optimiser message.
# - Verify the error against a tolerance of 1e-5 Hartree.
# - Show a labelled convergence plot.
#


## Advanced Task 3 – Convert H₂ into a qubit Hamiltonian

In Tasks 1 and 2, the Hamiltonian was already supplied as a list of Pauli terms. For a molecule, we must first create that Hamiltonian from the molecule's atoms and their positions. This task follows that conversion one step at a time.

### Step 1 – Describe the H₂ molecule

The molecule contains two hydrogen atoms. We place one at the origin and the other $0.74$ Å away. We also select STO-3G, a small **basis set**. A basis set is a collection of simple mathematical shapes used to describe where electrons may be found around the atoms.

Qiskit Nature provides [`PySCFDriver`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.drivers.PySCFDriver.html). It sends the atom names, positions, charge, spin, and basis set to the [PySCF](https://pyscf.org/user.html) chemistry package.

PySCF uses this information to calculate **molecular orbitals**. An orbital is a possible state that an electron can occupy in the molecule. PySCF also calculates the numerical coefficients needed to describe the energy of electrons in these orbitals and their interactions. Qiskit Nature collects the results in an `ElectronicStructureProblem`.

```python
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.74",
    basis="sto3g",
    charge=0,
    spin=0,
)
problem = driver.run()
```

At this point, `problem` contains the molecular information needed for the next step.

### Step 2 – Build a Hamiltonian for the electrons

The Hamiltonian must describe two things: which electron states are occupied and how those electrons contribute to the energy.

Electrons follow special rules because they are **fermions**. Qiskit Nature therefore stores the electron Hamiltonian as a [`FermionicOp`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.operators.FermionicOp.html). Each term describes electrons being placed into or removed from possible electron states, together with a coefficient that gives that term's contribution.

In the printed labels:

- `+_p` means place an electron in state $p$;
- `-_p` means remove an electron from state $p$.

The formal names for these actions are **creation** and **annihilation**. The [`second_q_op()`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.hamiltonians.ElectronicEnergy.html#qiskit_nature.second_q.hamiltonians.ElectronicEnergy.second_q_op) method builds the `FermionicOp` from the molecular information calculated in Step 1.

```python
fermionic_hamiltonian = problem.hamiltonian.second_q_op()
```

This Hamiltonian describes the electrons correctly, but it is not yet written as operations on qubits.

### Step 3 – Convert the Hamiltonian into Pauli operators

A quantum circuit works with qubits and operators such as $I$, $X$, $Y$, and $Z$. It cannot directly use the `+` and `-` electron operations from Step 2.

Qiskit Nature provides [`JordanWignerMapper`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.mappers.JordanWignerMapper.html) to perform the conversion. It uses one qubit to record whether each possible electron state is empty or occupied. Its `map()` method replaces the fermionic terms with weighted strings of Pauli operators.

```python
mapper = JordanWignerMapper()
qubit_hamiltonian = mapper.map(fermionic_hamiltonian)
```

The result is a Qiskit [`SparsePauliOp`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp). This is the same kind of Pauli-based Hamiltonian used in Tasks 1 and 2, although it now contains more terms because it was generated from a molecule.

### Step 4 – Use the qubit Hamiltonian in VQE

The conversion is now complete. Qiskit's `QuantumCircuit`, `ParameterVector`, and [`Statevector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector) are used to prepare and evaluate a trial quantum state. SciPy's [`minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html) changes the circuit parameters to reduce the energy.

The VQE circuit uses the familiar $R_y$ and `CX` gates from the earlier tasks. The only new part is that its Hamiltonian was generated from H₂ rather than supplied directly.

The complete workflow is

$$
\begin{array}{c}
\boxed{\text{H}_2\text{ atoms and positions}} \\
{\scriptstyle \text{molecular input}} \\[4pt]
\substack{\Big\downarrow \\ \texttt{PySCFDriver}} \\[8pt]
\boxed{\text{electron states and energy data}} \\
{\scriptstyle \texttt{ElectronicStructureProblem}} \\[4pt]
\substack{\Big\downarrow \\ \texttt{second\_q\_op()}} \\[8pt]
\boxed{\text{electron Hamiltonian}} \\
{\scriptstyle \texttt{FermionicOp}} \\[4pt]
\substack{\Big\downarrow \\ \text{Jordan--Wigner}} \\[8pt]
\boxed{\text{Pauli Hamiltonian}} \\
{\scriptstyle \texttt{SparsePauliOp}}
\end{array}
$$

The three checkpoints below let you inspect the result at each stage before moving on.

### References

1. [Qiskit Nature electronic-structure tutorial](https://qiskit-community.github.io/qiskit-nature/tutorials/01_electronic_structure.html)
2. [`PySCFDriver`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.drivers.PySCFDriver.html)
3. [`FermionicOp`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.operators.FermionicOp.html)
4. [`ElectronicEnergy.second_q_op`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.hamiltonians.ElectronicEnergy.html#qiskit_nature.second_q.hamiltonians.ElectronicEnergy.second_q_op)
5. [`JordanWignerMapper`](https://qiskit-community.github.io/qiskit-nature/stubs/qiskit_nature.second_q.mappers.JordanWignerMapper.html)
6. [Qiskit Nature mapper overview](https://qiskit-community.github.io/qiskit-nature/apidocs/qiskit_nature.second_q.mappers.html)

### VQE energy calculation

VQE calculates the expectation value of the qubit Hamiltonian:

$$
E(\boldsymbol\theta)
=\langle\psi(\boldsymbol\theta)|H_{\mathrm{qubit}}|\psi(\boldsymbol\theta)\rangle.
$$

In Qiskit, this value is calculated with `state.expectation_value(qubit_hamiltonian).real`.

In [ ]:
# Advanced Task 3 imports – run this cell
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper

print("Qiskit Nature conversion imports successful.")


In [ ]:
# Advanced Task 3 checkpoint 1 – molecule to electronic-structure problem
#
# Follow Step 1 of the workflow and its references in the markdown above.
#
# Instructions
# - Describe neutral H2 using two hydrogen atoms separated by 0.74 Angstrom.
# - Use the STO-3G basis and paired electron spins.
# - Run the molecular driver and store the electronic-structure problem.

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the electronic-structure problem object.


In [ ]:
# Advanced Task 3 checkpoint 2 – convert the Hamiltonian
#
# Follow Steps 2 and 3 of the workflow and their references above.
#
# Instructions
# - Construct the fermionic Hamiltonian from the electronic-structure problem.
# - Apply the Jordan–Wigner transformation to obtain the qubit Hamiltonian.
# - Store both objects for use in the next checkpoint.

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the fermionic Hamiltonian object.
# - Print the qubit Hamiltonian object.


In [ ]:
# Advanced Task 3 checkpoint 3 – inspect the mapped Hamiltonian and run VQE
#
# Follow Step 4 of the workflow and its references above.
#
# Instructions
# - Inspect the SparsePauliOp produced in checkpoint 2.
# - Build a generic parameterized circuit with two rotation layers and a
#   nearest-neighbour entangling layer between them.
# - Define an energy function that prepares the state and evaluates the mapped
#   qubit Hamiltonian.
# - Choose reproducible initial parameters.
# - Use COBYLA to minimise the energy and record the convergence history.
# - Plot energy against objective-function evaluation.

# YOUR CODE HERE

# REQUIRED OUTPUT
# - Print the SparsePauliOp type, qubit count, Pauli-term count, and first ten
#   Pauli terms.
# - Draw the circuit and print its qubit and parameter counts.
# - Print the initial and final electronic energies and optimiser message.
# - Print whether the optimisation lowered the energy.
# - Show one labelled convergence plot.
